In [1]:
# Retail Data Analytics Pipeline
# Objective
# Build a clean, analysis-ready dataset from U.S. Census retail data for downstream SQL analytics and dashboarding.

# Key Transformations
 # Multi-file ingestion (year-wise datasets)
 # Data cleaning and normalization
 # Wide-to-long transformation
 # Category standardization
 # Top-10 category segmentation
 # Data validation checks

In [2]:
import pandas as pd
import numpy as np
import glob
import os

In [3]:
# project folders
monthly_path = "../01_data/raw/*.csv"

In [4]:
# Step 1: Load Raw Data
# Read all yearly datasets and combine into a single dataframe.

monthly_files = glob.glob(monthly_path)

print("Total Monthly Files:", len(monthly_files))

monthly_files[:5]

Total Monthly Files: 34


['../01_data/raw\\data_1992_ts.csv',
 '../01_data/raw\\data_1993_ts.csv',
 '../01_data/raw\\data_1994_ts.csv',
 '../01_data/raw\\data_1995_ts.csv',
 '../01_data/raw\\data_1996_ts.csv']

In [5]:
sample = pd.read_csv(monthly_files[0])

sample.head()

,Estimates of Monthly Retail and Food Services Sales by Kind of Business: 1992,Unnamed: 1,Unnamed: 2,Unnamed: 3,Unnamed: 4,Unnamed: 5,Unnamed: 6,Unnamed: 7,Unnamed: 8,Unnamed: 9,Unnamed: 10,Unnamed: 11,Unnamed: 12,Unnamed: 13,Unnamed: 14,Unnamed: 15,Unnamed: 16
0,[Estimates are shown in millions of dollars an...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,NAICS Code,Kind of Business,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,NaN,NaN,Jan. 1992,Feb. 1992,Mar. 1992,Apr. 1992,May 1992,Jun. 1992,Jul. 1992,Aug. 1992,Sep. 1992,Oct. 1992,Nov. 1992,Dec. 1992,TOTAL,NaN,NaN
4,NaN,NOT ADJUSTED,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [6]:
sample.columns

Index(['Estimates of Monthly Retail and Food Services Sales by Kind of Business: 1992',
       'Unnamed: 1', 'Unnamed: 2', 'Unnamed: 3', 'Unnamed: 4', 'Unnamed: 5',
       'Unnamed: 6', 'Unnamed: 7', 'Unnamed: 8', 'Unnamed: 9', 'Unnamed: 10',
       'Unnamed: 11', 'Unnamed: 12', 'Unnamed: 13', 'Unnamed: 14',
       'Unnamed: 15', 'Unnamed: 16'],
      dtype='object')

In [7]:
sample.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 117 entries, 0 to 116
Data columns (total 17 columns):
 #   Column                                                                         Non-Null Count  Dtype  
---  ------                                                                         --------------  -----  
 0   Estimates of Monthly Retail and Food Services Sales by Kind of Business: 1992  97 non-null     object 
 1   Unnamed: 1                                                                     105 non-null    object 
 2   Unnamed: 2                                                                     103 non-null    object 
 3   Unnamed: 3                                                                     103 non-null    object 
 4   Unnamed: 4                                                                     103 non-null    object 
 5   Unnamed: 5                                                                     103 non-null    object 
 6   Unnamed: 6                

In [8]:
# Step 2: Data Cleaning Function
# Reusable function to clean and standardize each dataset.

In [9]:
df = pd.read_csv(monthly_files[0], skiprows=4)

df.head(10)

,Unnamed: 0,Unnamed: 1,Jan. 1992,Feb. 1992,Mar. 1992,Apr. 1992,May 1992,Jun. 1992,Jul. 1992,Aug. 1992,Sep. 1992,Oct. 1992,Nov. 1992,Dec. 1992,TOTAL,Unnamed: 15,Unnamed: 16
0,NaN,NOT ADJUSTED,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,NaN,"Retail and food services sales, total","1,42,051","1,42,498","1,54,439","1,58,430","1,64,788","1,63,417","1,64,575","1,65,387","1,59,892","1,68,529","1,66,918","2,03,616","19,54,540",NaN,NaN
2,NaN,Retail sales and food services excl motor vehi...,"1,13,046","1,12,155","1,20,284","1,23,509","1,29,318","1,26,010","1,27,756","1,30,891","1,24,924","1,32,773","1,35,297","1,71,343","15,47,306",NaN,NaN
3,NaN,Retail sales and food services excl gasoline s...,"1,30,133","1,31,091","1,42,350","1,46,173","1,51,538","1,50,188","1,50,904","1,51,763","1,46,941","1,55,061","1,54,039","1,90,315","18,00,496",NaN,NaN
4,NaN,Retail sales and food services excl motor vehi...,"1,01,128","1,00,748","1,08,195","1,11,252","1,16,068","1,12,781","1,14,085","1,17,267","1,11,973","1,19,305","1,22,418","1,58,042","13,93,262",NaN,NaN
5,NaN,"Retail sales, total","1,26,717","1,27,022","1,37,978","1,42,314","1,47,550","1,46,991","1,47,657","1,47,901","1,43,808","1,51,260","1,50,571","1,86,520","17,56,289",NaN,NaN
6,NaN,"Retail sales, total (excl. motor vehicle and p...","97,712","96,679","1,03,823","1,07,393","1,12,080","1,09,584","1,10,838","1,13,405","1,08,840","1,15,504","1,18,950","1,54,247","13,49,055",NaN,NaN
7,NaN,GAFO(1),"33,204","34,268","37,696","39,386","40,981","39,706","39,640","43,587","40,611","43,877","50,295","77,516","5,20,767",NaN,NaN
8,441,Motor vehicle and parts dealers,"29,005","30,343","34,155","34,921","35,470","37,407","36,819","34,496","34,968","35,756","31,621","32,273","4,07,234",NaN,NaN
9,"4411,4412",Automobile and other motor vehicle dealers,"26,074","27,421","30,809","31,494","32,002","33,798","33,195","30,979","31,529","32,080","28,273","28,928","3,66,582",NaN,NaN


In [10]:
df.columns = df.columns.str.strip()

df.head()

,Unnamed: 0,Unnamed: 1,Jan. 1992,Feb. 1992,Mar. 1992,Apr. 1992,May 1992,Jun. 1992,Jul. 1992,Aug. 1992,Sep. 1992,Oct. 1992,Nov. 1992,Dec. 1992,TOTAL,Unnamed: 15,Unnamed: 16
0,NaN,NOT ADJUSTED,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,NaN,"Retail and food services sales, total","1,42,051","1,42,498","1,54,439","1,58,430","1,64,788","1,63,417","1,64,575","1,65,387","1,59,892","1,68,529","1,66,918","2,03,616","19,54,540",NaN,NaN
2,NaN,Retail sales and food services excl motor vehi...,"1,13,046","1,12,155","1,20,284","1,23,509","1,29,318","1,26,010","1,27,756","1,30,891","1,24,924","1,32,773","1,35,297","1,71,343","15,47,306",NaN,NaN
3,NaN,Retail sales and food services excl gasoline s...,"1,30,133","1,31,091","1,42,350","1,46,173","1,51,538","1,50,188","1,50,904","1,51,763","1,46,941","1,55,061","1,54,039","1,90,315","18,00,496",NaN,NaN
4,NaN,Retail sales and food services excl motor vehi...,"1,01,128","1,00,748","1,08,195","1,11,252","1,16,068","1,12,781","1,14,085","1,17,267","1,11,973","1,19,305","1,22,418","1,58,042","13,93,262",NaN,NaN


In [11]:
df = df.dropna(axis=1, how="all")

df.head()

,Unnamed: 0,Unnamed: 1,Jan. 1992,Feb. 1992,Mar. 1992,Apr. 1992,May 1992,Jun. 1992,Jul. 1992,Aug. 1992,Sep. 1992,Oct. 1992,Nov. 1992,Dec. 1992,TOTAL
0,NaN,NOT ADJUSTED,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,NaN,"Retail and food services sales, total","1,42,051","1,42,498","1,54,439","1,58,430","1,64,788","1,63,417","1,64,575","1,65,387","1,59,892","1,68,529","1,66,918","2,03,616","19,54,540"
2,NaN,Retail sales and food services excl motor vehi...,"1,13,046","1,12,155","1,20,284","1,23,509","1,29,318","1,26,010","1,27,756","1,30,891","1,24,924","1,32,773","1,35,297","1,71,343","15,47,306"
3,NaN,Retail sales and food services excl gasoline s...,"1,30,133","1,31,091","1,42,350","1,46,173","1,51,538","1,50,188","1,50,904","1,51,763","1,46,941","1,55,061","1,54,039","1,90,315","18,00,496"
4,NaN,Retail sales and food services excl motor vehi...,"1,01,128","1,00,748","1,08,195","1,11,252","1,16,068","1,12,781","1,14,085","1,17,267","1,11,973","1,19,305","1,22,418","1,58,042","13,93,262"


In [12]:
df = df.rename(columns={
    df.columns[0]: "naics_code",
    df.columns[1]: "business_category"
})

df.head()

,naics_code,business_category,Jan. 1992,Feb. 1992,Mar. 1992,Apr. 1992,May 1992,Jun. 1992,Jul. 1992,Aug. 1992,Sep. 1992,Oct. 1992,Nov. 1992,Dec. 1992,TOTAL
0,NaN,NOT ADJUSTED,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,NaN,"Retail and food services sales, total","1,42,051","1,42,498","1,54,439","1,58,430","1,64,788","1,63,417","1,64,575","1,65,387","1,59,892","1,68,529","1,66,918","2,03,616","19,54,540"
2,NaN,Retail sales and food services excl motor vehi...,"1,13,046","1,12,155","1,20,284","1,23,509","1,29,318","1,26,010","1,27,756","1,30,891","1,24,924","1,32,773","1,35,297","1,71,343","15,47,306"
3,NaN,Retail sales and food services excl gasoline s...,"1,30,133","1,31,091","1,42,350","1,46,173","1,51,538","1,50,188","1,50,904","1,51,763","1,46,941","1,55,061","1,54,039","1,90,315","18,00,496"
4,NaN,Retail sales and food services excl motor vehi...,"1,01,128","1,00,748","1,08,195","1,11,252","1,16,068","1,12,781","1,14,085","1,17,267","1,11,973","1,19,305","1,22,418","1,58,042","13,93,262"


In [13]:
df = df[df["business_category"].notna()]

df.head(15)

,naics_code,business_category,Jan. 1992,Feb. 1992,Mar. 1992,Apr. 1992,May 1992,Jun. 1992,Jul. 1992,Aug. 1992,Sep. 1992,Oct. 1992,Nov. 1992,Dec. 1992,TOTAL
0,NaN,NOT ADJUSTED,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,NaN,"Retail and food services sales, total","1,42,051","1,42,498","1,54,439","1,58,430","1,64,788","1,63,417","1,64,575","1,65,387","1,59,892","1,68,529","1,66,918","2,03,616","19,54,540"
2,NaN,Retail sales and food services excl motor vehi...,"1,13,046","1,12,155","1,20,284","1,23,509","1,29,318","1,26,010","1,27,756","1,30,891","1,24,924","1,32,773","1,35,297","1,71,343","15,47,306"
3,NaN,Retail sales and food services excl gasoline s...,"1,30,133","1,31,091","1,42,350","1,46,173","1,51,538","1,50,188","1,50,904","1,51,763","1,46,941","1,55,061","1,54,039","1,90,315","18,00,496"
4,NaN,Retail sales and food services excl motor vehi...,"1,01,128","1,00,748","1,08,195","1,11,252","1,16,068","1,12,781","1,14,085","1,17,267","1,11,973","1,19,305","1,22,418","1,58,042","13,93,262"
5,NaN,"Retail sales, total","1,26,717","1,27,022","1,37,978","1,42,314","1,47,550","1,46,991","1,47,657","1,47,901","1,43,808","1,51,260","1,50,571","1,86,520","17,56,289"
6,NaN,"Retail sales, total (excl. motor vehicle and p...","97,712","96,679","1,03,823","1,07,393","1,12,080","1,09,584","1,10,838","1,13,405","1,08,840","1,15,504","1,18,950","1,54,247","13,49,055"
7,NaN,GAFO(1),"33,204","34,268","37,696","39,386","40,981","39,706","39,640","43,587","40,611","43,877","50,295","77,516","5,20,767"
8,441,Motor vehicle and parts dealers,"29,005","30,343","34,155","34,921","35,470","37,407","36,819","34,496","34,968","35,756","31,621","32,273","4,07,234"
9,"4411,4412",Automobile and other motor vehicle dealers,"26,074","27,421","30,809","31,494","32,002","33,798","33,195","30,979","31,529","32,080","28,273","28,928","3,66,582"


In [14]:
df = df.drop(columns=["TOTAL"], errors="ignore")

df.head()

,naics_code,business_category,Jan. 1992,Feb. 1992,Mar. 1992,Apr. 1992,May 1992,Jun. 1992,Jul. 1992,Aug. 1992,Sep. 1992,Oct. 1992,Nov. 1992,Dec. 1992
0,NaN,NOT ADJUSTED,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,NaN,"Retail and food services sales, total","1,42,051","1,42,498","1,54,439","1,58,430","1,64,788","1,63,417","1,64,575","1,65,387","1,59,892","1,68,529","1,66,918","2,03,616"
2,NaN,Retail sales and food services excl motor vehi...,"1,13,046","1,12,155","1,20,284","1,23,509","1,29,318","1,26,010","1,27,756","1,30,891","1,24,924","1,32,773","1,35,297","1,71,343"
3,NaN,Retail sales and food services excl gasoline s...,"1,30,133","1,31,091","1,42,350","1,46,173","1,51,538","1,50,188","1,50,904","1,51,763","1,46,941","1,55,061","1,54,039","1,90,315"
4,NaN,Retail sales and food services excl motor vehi...,"1,01,128","1,00,748","1,08,195","1,11,252","1,16,068","1,12,781","1,14,085","1,17,267","1,11,973","1,19,305","1,22,418","1,58,042"


In [15]:
df_long = df.melt(
    id_vars=["naics_code", "business_category"],
    var_name="month",
    value_name="sales"
)

df_long.head(10)

,naics_code,business_category,month,sales
0,NaN,NOT ADJUSTED,Jan. 1992,NaN
1,NaN,"Retail and food services sales, total",Jan. 1992,"1,42,051"
2,NaN,Retail sales and food services excl motor vehi...,Jan. 1992,"1,13,046"
3,NaN,Retail sales and food services excl gasoline s...,Jan. 1992,"1,30,133"
4,NaN,Retail sales and food services excl motor vehi...,Jan. 1992,"1,01,128"
5,NaN,"Retail sales, total",Jan. 1992,"1,26,717"
6,NaN,"Retail sales, total (excl. motor vehicle and p...",Jan. 1992,"97,712"
7,NaN,GAFO(1),Jan. 1992,"33,204"
8,441,Motor vehicle and parts dealers,Jan. 1992,"29,005"
9,"4411,4412",Automobile and other motor vehicle dealers,Jan. 1992,"26,074"


In [16]:
df_long["sales"] = (
    df_long["sales"]
    .astype(str)
    .str.replace(",", "")
)

df_long["sales"] = pd.to_numeric(df_long["sales"], errors="coerce")

df_long.head()

,naics_code,business_category,month,sales
0,NaN,NOT ADJUSTED,Jan. 1992,NaN
1,NaN,"Retail and food services sales, total",Jan. 1992,142051.0
2,NaN,Retail sales and food services excl motor vehi...,Jan. 1992,113046.0
3,NaN,Retail sales and food services excl gasoline s...,Jan. 1992,130133.0
4,NaN,Retail sales and food services excl motor vehi...,Jan. 1992,101128.0


In [17]:
df_long["month"] = df_long["month"].str.replace(".", "", regex=False)

df_long["sales_date"] = pd.to_datetime(df_long["month"], format="%b %Y")

df_long.head()

,naics_code,business_category,month,sales,sales_date
0,NaN,NOT ADJUSTED,Jan 1992,NaN,1992-01-01
1,NaN,"Retail and food services sales, total",Jan 1992,142051.0,1992-01-01
2,NaN,Retail sales and food services excl motor vehi...,Jan 1992,113046.0,1992-01-01
3,NaN,Retail sales and food services excl gasoline s...,Jan 1992,130133.0,1992-01-01
4,NaN,Retail sales and food services excl motor vehi...,Jan 1992,101128.0,1992-01-01


In [18]:
df_long = df_long[[
    "sales_date",
    "naics_code",
    "business_category",
    "sales"
]]

df_long.head()

,sales_date,naics_code,business_category,sales
0,1992-01-01,NaN,NOT ADJUSTED,NaN
1,1992-01-01,NaN,"Retail and food services sales, total",142051.0
2,1992-01-01,NaN,Retail sales and food services excl motor vehi...,113046.0
3,1992-01-01,NaN,Retail sales and food services excl gasoline s...,130133.0
4,1992-01-01,NaN,Retail sales and food services excl motor vehi...,101128.0


In [19]:
df_long = df_long.dropna(subset=["sales"])

df_long.head()

,sales_date,naics_code,business_category,sales
1,1992-01-01,NaN,"Retail and food services sales, total",142051.0
2,1992-01-01,NaN,Retail sales and food services excl motor vehi...,113046.0
3,1992-01-01,NaN,Retail sales and food services excl gasoline s...,130133.0
4,1992-01-01,NaN,Retail sales and food services excl motor vehi...,101128.0
5,1992-01-01,NaN,"Retail sales, total",126717.0


In [20]:
df_long.head()
df_long.shape

(1176, 4)

In [21]:
def clean_retail_file(file_path):

    # load dataset
    df = pd.read_csv(file_path, skiprows=4)

    # remove empty columns
    df = df.dropna(axis=1, how="all")

    # rename first two columns
    df = df.rename(columns={
        df.columns[0]: "naics_code",
        df.columns[1]: "business_category"
    })

    # remove rows without category
    df = df[df["business_category"].notna()]

    # remove TOTAL column if present
    df = df.drop(columns=["TOTAL"], errors="ignore")

    # reshape wide → long
    df_long = df.melt(
        id_vars=["naics_code", "business_category"],
        var_name="month",
        value_name="sales"
    )

    # clean sales numbers
    df_long["sales"] = (
        df_long["sales"]
        .astype(str)
        .str.replace(",", "")
    )

    df_long["sales"] = pd.to_numeric(df_long["sales"], errors="coerce")

    # clean month column
    df_long["month"] = (
        df_long["month"]
        .astype(str)
        .str.replace(".", "", regex=False)
        .str.replace(r"\(.*\)", "", regex=True)
        .str.strip()
    )

    # keep only valid months
    df_long = df_long[df_long["month"].str.contains(r"^[A-Za-z]{3} \d{4}$", na=False)]

    # convert to date
    df_long["sales_date"] = pd.to_datetime(df_long["month"], format="%b %Y")

    # final columns
    df_long = df_long[[
        "sales_date",
        "naics_code",
        "business_category",
        "sales"
    ]]

    # remove missing sales
    df_long = df_long.dropna(subset=["sales"])

    return df_long 

In [22]:
all_data = []

for file in monthly_files:
    
    cleaned = clean_retail_file(file)
    
    all_data.append(cleaned)

retail_df = pd.concat(all_data, ignore_index=True)

retail_df.shape 

(40555, 4)

In [23]:
retail_df.head()

,sales_date,naics_code,business_category,sales
0,1992-01-01,NaN,"Retail and food services sales, total",142051.0
1,1992-01-01,NaN,Retail sales and food services excl motor vehi...,113046.0
2,1992-01-01,NaN,Retail sales and food services excl gasoline s...,130133.0
3,1992-01-01,NaN,Retail sales and food services excl motor vehi...,101128.0
4,1992-01-01,NaN,"Retail sales, total",126717.0


In [24]:
retail_df.tail()

,sales_date,naics_code,business_category,sales
40550,2025-12-01,453,Miscellaneous stores retailers,15452.0
40551,2025-12-01,454,Nonstore retailers,130499.0
40552,2025-12-01,4541,Electronic shopping and mail order houses,122809.0
40553,2025-12-01,45431,Fuel dealers,3587.0
40554,2025-12-01,722,Food services and drinking places,100012.0


In [25]:
retail_df["naics_code"] = retail_df["naics_code"].ffill()

In [26]:
retail_df.head(20) 

,sales_date,naics_code,business_category,sales
0,1992-01-01,NaN,"Retail and food services sales, total",142051.0
1,1992-01-01,NaN,Retail sales and food services excl motor vehi...,113046.0
2,1992-01-01,NaN,Retail sales and food services excl gasoline s...,130133.0
3,1992-01-01,NaN,Retail sales and food services excl motor vehi...,101128.0
4,1992-01-01,NaN,"Retail sales, total",126717.0
5,1992-01-01,NaN,"Retail sales, total (excl. motor vehicle and p...",97712.0
6,1992-01-01,NaN,GAFO(1),33204.0
7,1992-01-01,441,Motor vehicle and parts dealers,29005.0
8,1992-01-01,"4411,4412",Automobile and other motor vehicle dealers,26074.0
9,1992-01-01,4411,Automobile dealers,25185.0


In [27]:
retail_df["naics_code"] = (
    retail_df["naics_code"]
    .astype(str)
    .str.replace(",", "_")
    .str.strip()
)

In [28]:
retail_df.head(20) 

,sales_date,naics_code,business_category,sales
0,1992-01-01,nan,"Retail and food services sales, total",142051.0
1,1992-01-01,nan,Retail sales and food services excl motor vehi...,113046.0
2,1992-01-01,nan,Retail sales and food services excl gasoline s...,130133.0
3,1992-01-01,nan,Retail sales and food services excl motor vehi...,101128.0
4,1992-01-01,nan,"Retail sales, total",126717.0
5,1992-01-01,nan,"Retail sales, total (excl. motor vehicle and p...",97712.0
6,1992-01-01,nan,GAFO(1),33204.0
7,1992-01-01,441,Motor vehicle and parts dealers,29005.0
8,1992-01-01,4411_4412,Automobile and other motor vehicle dealers,26074.0
9,1992-01-01,4411,Automobile dealers,25185.0


In [29]:
retail_df.info() 

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 40555 entries, 0 to 40554
Data columns (total 4 columns):
 #   Column             Non-Null Count  Dtype         
---  ------             --------------  -----         
 0   sales_date         40555 non-null  datetime64[ns]
 1   naics_code         40555 non-null  object        
 2   business_category  40555 non-null  object        
 3   sales              40555 non-null  float64       
dtypes: datetime64[ns](1), float64(1), object(2)
memory usage: 1.2+ MB


In [30]:
retail_df.shape 

(40555, 4)

In [31]:
retail_df.head(10)

,sales_date,naics_code,business_category,sales
0,1992-01-01,nan,"Retail and food services sales, total",142051.0
1,1992-01-01,nan,Retail sales and food services excl motor vehi...,113046.0
2,1992-01-01,nan,Retail sales and food services excl gasoline s...,130133.0
3,1992-01-01,nan,Retail sales and food services excl motor vehi...,101128.0
4,1992-01-01,nan,"Retail sales, total",126717.0
5,1992-01-01,nan,"Retail sales, total (excl. motor vehicle and p...",97712.0
6,1992-01-01,nan,GAFO(1),33204.0
7,1992-01-01,441,Motor vehicle and parts dealers,29005.0
8,1992-01-01,4411_4412,Automobile and other motor vehicle dealers,26074.0
9,1992-01-01,4411,Automobile dealers,25185.0


In [32]:
# Step 3: Category Standardization
# Map raw business categories into meaningful analytical groups.

In [33]:
category_mapping = {

# Automotive
"Automobile and other motor vehicle dealers": "Auto Dealers",
"Automobile dealers": "Auto Dealers",
"New car dealers": "Auto Dealers",
"Used car dealers": "Auto Dealers",
"Motor vehicle and parts dealers": "Auto Retail",
"Automotive parts acc. and tire stores": "Auto Parts",

# Apparel
"Clothing and clothing access. stores": "Apparel",
"Clothing stores": "Apparel",
"Men's clothing stores": "Men's Apparel",
"Women's clothing stores": "Women's Apparel",
"Family clothing stores": "Family Apparel",
"Other clothing stores": "Apparel Specialty",
"Shoe stores": "Footwear",
"Jewelry stores": "Jewelry",

# Home & Furniture
"Furniture and home furnishings stores": "Furniture",
"Furniture stores": "Furniture",
"Home furnishings stores": "Home Furnishings",
"All other home furnishings stores": "Home Furnishings",
"Floor covering stores": "Home Furnishings",

# Electronics
"Electronics and appliance stores": "Electronics",
"Electronics stores": "Electronics",
"Household appliance stores": "Electronics",

# Food Retail
"Grocery stores": "Grocery",
"Supermarkets and other grocery (except convenience) stores": "Supermarkets",
"Food and beverage stores": "Food Retail",
"Beer wine and liquor stores": "Liquor",

# General Retail
"General merchandise stores": "General Retail",
"Department stores": "Department Stores",
"Warehouse clubs and supercenters": "Warehouse Clubs",
"All other gen. merchandise stores": "General Retail",

# Home Improvement
"Building mat. and garden equip. and supplies dealers": "Building Materials",
"Building mat. and supplies dealers": "Building Materials",
"Hardware stores": "Hardware",
"Paint and wallpaper stores": "Paint & Decor",

# Hobby / Books
"Sporting goods hobby musical instrument and book stores": "Sports & Hobby",
"Sporting goods stores": "Sports Retail",
"Hobby toy and game stores": "Toys & Games",
"Book stores": "Book Retail",

# Health
"Health and personal care stores": "Health & Personal Care",
"Pharmacies and drug stores": "Pharmacy",

# Ecommerce
"Electronic shopping and mail order houses": "E-Commerce",
"Electronic shopping and mail-order houses": "E-Commerce",
"Nonstore retailers": "E-Commerce",

# Fuel
"Fuel dealers": "Fuel Retail",
"Gasoline stations": "Fuel Retail",

# Food Service
"Restaurants and other eating places": "Restaurants",
"Full service restaurants": "Restaurants",
"Limited service eating places": "Fast Food",
"Food services and drinking places": "Food Service",
"Drinking places": "Bars",

# Specialty Retail
"Office supplies stationery and gift stores": "Office & Gifts",
"Office supplies and stationery stores": "Office Supplies",
"Gift novelty and souvenir stores": "Gift Shops",

# Resale
"Used merchandise stores": "Secondhand Retail",

# Misc
"Miscellaneous store retailers": "Specialty Retail",
"Miscellaneous stores retailers": "Specialty Retail",

# Retail Indicators
"GAFO(1)": "GAFO",

# Retail aggregates
"Retail sales total": "Retail Total",
"Retail and food services sales total": "Retail & Food Total",
"Retail sales and food services excl gasoline stations": "Retail ex Fuel",
"Retail sales and food services excl motor vehicle and parts": "Retail ex Auto",
"Retail sales total (excl. motor vehicle and parts dealers)": "Retail ex Auto",
"Retail sales and food services excl motor vehicle and parts and gasoline stations": "Retail ex Auto & Fuel",
"Furniture, home furn, electronics, and appliance stores": "Furniture & Electronics",
"Gen. merchandise stores incl. warehouse clubs & supercenters": "General Retail"
}

In [34]:
import re

#  — normalize business_category
retail_df["business_category_clean"] = (
    retail_df["business_category"]
    .str.lower()
    .str.replace(",", "")
    .str.strip()
)

#  — create mapping with normalized keys
category_mapping_clean = {
    re.sub(r"\s+", " ", key.lower().replace(",", "")).strip(): value
    for key, value in category_mapping.items()
}

#  — map to professional short name
retail_df["category_short_name"] = retail_df["business_category_clean"].map(category_mapping_clean)

#  — fallback: if not mapped, keep original category
retail_df["category_short_name"] = retail_df["category_short_name"].fillna(retail_df["business_category"])

#  — drop helper column if needed
retail_df.drop(columns=["business_category_clean"], inplace=True)

In [35]:
retail_df.head(10)

,sales_date,naics_code,business_category,sales,category_short_name
0,1992-01-01,nan,"Retail and food services sales, total",142051.0,Retail & Food Total
1,1992-01-01,nan,Retail sales and food services excl motor vehi...,113046.0,Retail ex Auto
2,1992-01-01,nan,Retail sales and food services excl gasoline s...,130133.0,Retail ex Fuel
3,1992-01-01,nan,Retail sales and food services excl motor vehi...,101128.0,Retail ex Auto & Fuel
4,1992-01-01,nan,"Retail sales, total",126717.0,Retail Total
5,1992-01-01,nan,"Retail sales, total (excl. motor vehicle and p...",97712.0,Retail ex Auto
6,1992-01-01,nan,GAFO(1),33204.0,GAFO
7,1992-01-01,441,Motor vehicle and parts dealers,29005.0,Auto Retail
8,1992-01-01,4411_4412,Automobile and other motor vehicle dealers,26074.0,Auto Dealers
9,1992-01-01,4411,Automobile dealers,25185.0,Auto Dealers


In [36]:
retail_df[retail_df["category_short_name"].isna()]["business_category"].unique()

array([], dtype=object)

In [37]:
retail_df["category_short_name"].unique() 

array(['Retail & Food Total', 'Retail ex Auto', 'Retail ex Fuel',
       'Retail ex Auto & Fuel', 'Retail Total', 'GAFO', 'Auto Retail',
       'Auto Dealers', 'Auto Parts', 'Furniture & Electronics',
       'Furniture', 'Home Furnishings', 'Electronics',
       'Building Materials', 'Hardware', 'Food Retail', 'Grocery',
       'Liquor', 'Health & Personal Care', 'Pharmacy', 'Fuel Retail',
       'Apparel', "Men's Apparel", "Women's Apparel", 'Family Apparel',
       'Footwear', 'Jewelry', 'Sports & Hobby', 'Sports Retail',
       'Toys & Games', 'Book Retail', 'General Retail',
       'Department Stores', 'Warehouse Clubs', 'Specialty Retail',
       'Office & Gifts', 'Office Supplies', 'Gift Shops',
       'Secondhand Retail', 'E-Commerce', 'Food Service', 'Bars',
       'Restaurants', 'Fast Food', 'Paint & Decor', 'Supermarkets',
       'Apparel Specialty'], dtype=object)

In [38]:
import pandas as pd
pd.set_option('display.max_rows', None)

In [39]:
retail_df["category_short_name"].value_counts()

category_short_name
General Retail             2448
Auto Dealers               2040
Electronics                1632
Building Materials         1632
E-Commerce                 1632
Apparel                    1632
Retail ex Auto             1632
Fuel Retail                1632
Furniture                  1224
Department Stores          1224
Home Furnishings            884
Auto Retail                 816
GAFO                        816
Retail Total                816
Retail & Food Total         816
Sports & Hobby              816
Furniture & Electronics     816
Auto Parts                  816
Retail ex Fuel              816
Retail ex Auto & Fuel       816
Health & Personal Care      816
Food Retail                 816
Grocery                     816
Warehouse Clubs             816
Footwear                    816
Specialty Retail            816
Food Service                816
Pharmacy                    816
Liquor                      816
Women's Apparel             816
Restaurants         

In [40]:
sorted(retail_df["category_short_name"].unique()) 

['Apparel',
 'Apparel Specialty',
 'Auto Dealers',
 'Auto Parts',
 'Auto Retail',
 'Bars',
 'Book Retail',
 'Building Materials',
 'Department Stores',
 'E-Commerce',
 'Electronics',
 'Family Apparel',
 'Fast Food',
 'Food Retail',
 'Food Service',
 'Footwear',
 'Fuel Retail',
 'Furniture',
 'Furniture & Electronics',
 'GAFO',
 'General Retail',
 'Gift Shops',
 'Grocery',
 'Hardware',
 'Health & Personal Care',
 'Home Furnishings',
 'Jewelry',
 'Liquor',
 "Men's Apparel",
 'Office & Gifts',
 'Office Supplies',
 'Paint & Decor',
 'Pharmacy',
 'Restaurants',
 'Retail & Food Total',
 'Retail Total',
 'Retail ex Auto',
 'Retail ex Auto & Fuel',
 'Retail ex Fuel',
 'Secondhand Retail',
 'Specialty Retail',
 'Sports & Hobby',
 'Sports Retail',
 'Supermarkets',
 'Toys & Games',
 'Warehouse Clubs',
 "Women's Apparel"]

In [41]:
retail_df["category_short_name"].nunique()

47

In [42]:
category_sales = (
    retail_df.groupby("category_short_name")["sales"]
    .sum()
    .sort_values(ascending=False) 
)

In [43]:
category_share = (category_sales / category_sales.sum()) * 100

result = category_share.reset_index()
result.columns = ["category", "market_share_percent"]

result.head(20) 

,category,market_share_percent
0,Retail ex Auto,19.897133
1,Retail & Food Total,13.447379
2,Retail ex Fuel,12.272769
3,Retail Total,11.947188
4,Retail ex Auto & Fuel,9.524053
5,Auto Dealers,4.873735
6,GAFO,3.125244
7,General Retail,3.041485
8,Auto Retail,2.748717
9,E-Commerce,2.474878


In [44]:
retail_df.columns 

Index(['sales_date', 'naics_code', 'business_category', 'sales',
       'category_short_name'],
      dtype='object')

In [45]:
retail_df.groupby("category_short_name")["sales"].sum() 

category_short_name
Apparel                     24274012.0
Apparel Specialty             329953.0
Auto Dealers               112902324.0
Auto Parts                   5334129.0
Auto Retail                 63675303.0
Bars                          425960.0
Book Retail                   402734.0
Building Materials          38063344.0
Department Stores           15674884.0
E-Commerce                  57331703.0
Electronics                  9084702.0
Family Apparel               2764584.0
Fast Food                    7703108.0
Food Retail                 40875299.0
Food Service                34752623.0
Footwear                     1933501.0
Fuel Retail                 29308207.0
Furniture                    8389612.0
Furniture & Electronics     12633064.0
GAFO                        72397718.0
General Retail              70457410.0
Gift Shops                    528879.0
Grocery                     36971746.0
Hardware                      753245.0
Health & Personal Care      16712837.0
Home 

In [46]:
retail_df.groupby("category_short_name")["sales"].sum().sort_values(ascending=False)

category_short_name
Retail ex Auto             460926309.0
Retail & Food Total        311514769.0
Retail ex Fuel             284304396.0
Retail Total               276762146.0
Retail ex Auto & Fuel      220629093.0
Auto Dealers               112902324.0
GAFO                        72397718.0
General Retail              70457410.0
Auto Retail                 63675303.0
E-Commerce                  57331703.0
Food Retail                 40875299.0
Building Materials          38063344.0
Grocery                     36971746.0
Food Service                34752623.0
Fuel Retail                 29308207.0
Warehouse Clubs             24389286.0
Apparel                     24274012.0
Restaurants                 20474841.0
Health & Personal Care      16712837.0
Department Stores           15674884.0
Supermarkets                14580602.0
Pharmacy                    14256913.0
Furniture & Electronics     12633064.0
Electronics                  9084702.0
Furniture                    8389612.0
Fast 

In [47]:
retail_df["category_short_name"].nunique()

47

In [48]:
# Step 4: Top-10 Category Segmentation
# Identify dominant retail categories and group remaining into "Other Retail Sectors".

In [49]:
# Group by category and sum sales
category_sales = retail_df.groupby("category_short_name")["sales"].sum().sort_values(ascending=False)

# Get Top 10 categories
top10 = category_sales.head(10).index.tolist()

# Create Top-10 + Other Retail Sectors column
retail_df["category_top10"] = retail_df["category_short_name"].apply(
    lambda x: x if x in top10 else "Other Retail Sectors")

# Year 
retail_df["sales_year"] = retail_df["sales_date"].dt.year

# Extract short month name (Jan, Feb, Mar, ...)
retail_df["sales_month"] = retail_df["sales_date"].dt.strftime("%m-%b")

# Create YYYY-MonthName format (e.g., 2025-Jan)
retail_df["year_month"] = retail_df["sales_date"].dt.strftime("%b %Y")




In [50]:
# Step 5: Final Dataset Export
# Save cleaned dataset for PostgreSQL and Metabase dashboards.

# Optional: Verify distribution
print(retail_df["category_top10"].value_counts())
retail_df = retail_df[
    [
        "sales_date",
        "sales_year",
        "sales_month",
        "year_month",
        "business_category",
        "category_short_name",
        "category_top10",
        "sales"
    ]
]

# Save final CSV for PostgreSQL / Metabase
retail_df.to_csv("retail_data1.csv", index=False)

category_top10
Other Retail Sectors     27907
General Retail            2448
Auto Dealers              2040
Retail ex Auto            1632
E-Commerce                1632
Retail ex Fuel             816
Retail & Food Total        816
Auto Retail                816
GAFO                       816
Retail Total               816
Retail ex Auto & Fuel      816
Name: count, dtype: int64


In [51]:
# Check shape and basic info
print("Dataset shape:", retail_df.shape)
print("\nColumns and types:")
print(retail_df.dtypes)

# Check for missing values
print("\nMissing values per column:")
print(retail_df.isnull().sum()) 

Dataset shape: (40555, 8)

Columns and types:
sales_date             datetime64[ns]
sales_year                      int32
sales_month                    object
year_month                     object
business_category              object
category_short_name            object
category_top10                 object
sales                         float64
dtype: object

Missing values per column:
sales_date             0
sales_year             0
sales_month            0
year_month             0
business_category      0
category_short_name    0
category_top10         0
sales                  0
dtype: int64


In [52]:
# Unique categories in Top-10 + Other
print("\nTop-10 + Other Retail Sectors categories:")
print(retail_df["category_top10"].value_counts())

# Unique original categories
print("\nOriginal 47 categories:")
print(retail_df["category_short_name"].nunique()) 


Top-10 + Other Retail Sectors categories:
category_top10
Other Retail Sectors     27907
General Retail            2448
Auto Dealers              2040
Retail ex Auto            1632
E-Commerce                1632
Retail ex Fuel             816
Retail & Food Total        816
Auto Retail                816
GAFO                       816
Retail Total               816
Retail ex Auto & Fuel      816
Name: count, dtype: int64

Original 47 categories:
47


In [53]:
# Summary statistics
print("\nSales summary:")
print(retail_df["sales"].describe())

# Check for negative or zero sales
negative_sales = retail_df[retail_df["sales"] <= 0]
print("\nNegative or zero sales rows:", negative_sales.shape[0]) 


Sales summary:
count     40555.000000
mean      57121.103982
std      110559.762051
min          81.000000
25%        3494.000000
50%       13944.000000
75%       48293.500000
max      816616.000000
Name: sales, dtype: float64

Negative or zero sales rows: 0


In [54]:
# Ensure all non-top10 categories are grouped
top10 = retail_df.groupby("category_short_name")["sales"].sum().sort_values(ascending=False).head(10).index.tolist()

incorrect_mapping = retail_df[
    (retail_df["category_short_name"].isin(top10) == False) &
    (retail_df["category_top10"] != "Other Retail Sectors")
]

print("\nIncorrectly mapped categories (should be 0):", incorrect_mapping.shape[0]) 


Incorrectly mapped categories (should be 0): 0


In [55]:
# Check Top-10 + Other revenue
top10_sales = retail_df.groupby("category_top10")["sales"].sum().sort_values(ascending=False)
print("\nRevenue by Top-10 + Other Retail Sectors:")
print(top10_sales)


Revenue by Top-10 + Other Retail Sectors:
category_top10
Retail ex Auto           460926309.0
Other Retail Sectors     385645201.0
Retail & Food Total      311514769.0
Retail ex Fuel           284304396.0
Retail Total             276762146.0
Retail ex Auto & Fuel    220629093.0
Auto Dealers             112902324.0
GAFO                      72397718.0
General Retail            70457410.0
Auto Retail               63675303.0
E-Commerce                57331703.0
Name: sales, dtype: float64


In [56]:
retail_df.head(20)

,sales_date,sales_year,sales_month,year_month,business_category,category_short_name,category_top10,sales
0,1992-01-01,1992,01-Jan,Jan 1992,"Retail and food services sales, total",Retail & Food Total,Retail & Food Total,142051.0
1,1992-01-01,1992,01-Jan,Jan 1992,Retail sales and food services excl motor vehi...,Retail ex Auto,Retail ex Auto,113046.0
2,1992-01-01,1992,01-Jan,Jan 1992,Retail sales and food services excl gasoline s...,Retail ex Fuel,Retail ex Fuel,130133.0
3,1992-01-01,1992,01-Jan,Jan 1992,Retail sales and food services excl motor vehi...,Retail ex Auto & Fuel,Retail ex Auto & Fuel,101128.0
4,1992-01-01,1992,01-Jan,Jan 1992,"Retail sales, total",Retail Total,Retail Total,126717.0
5,1992-01-01,1992,01-Jan,Jan 1992,"Retail sales, total (excl. motor vehicle and p...",Retail ex Auto,Retail ex Auto,97712.0
6,1992-01-01,1992,01-Jan,Jan 1992,GAFO(1),GAFO,GAFO,33204.0
7,1992-01-01,1992,01-Jan,Jan 1992,Motor vehicle and parts dealers,Auto Retail,Auto Retail,29005.0
8,1992-01-01,1992,01-Jan,Jan 1992,Automobile and other motor vehicle dealers,Auto Dealers,Auto Dealers,26074.0
9,1992-01-01,1992,01-Jan,Jan 1992,Automobile dealers,Auto Dealers,Auto Dealers,25185.0
